In [1]:
import sys
sys.path.append("..")

from src.embeddings import NILCEmbeddings
from src.vectorstore import carregar_colecao
from src.retrieval import processar_pergunta, recuperar, imprimir_evidencias

store = carregar_colecao(NILCEmbeddings())

In [2]:
perguntas = [
    "Quais são os principais problemas relatados pelos clientes?",
    "Quais reclamações estão relacionadas à entrega?",
    "O que os clientes mais elogiam nos produtos?",
    "Quais problemas aparecem nas avaliações de clientes insatisfeitos?",
    "Existem padrões de reclamação sobre a qualidade dos produtos?",
    "Qual foi o desempenho financeiro da Olist em 2018?",  # fora do escopo, de propósito
]

for p in perguntas:
    print(f"{p}\n   -> {processar_pergunta(p)}\n")

Quais são os principais problemas relatados pelos clientes?
   -> principais problemas relatados clientes

Quais reclamações estão relacionadas à entrega?
   -> reclamações relacionadas à entrega

O que os clientes mais elogiam nos produtos?
   -> clientes mais elogiam produtos

Quais problemas aparecem nas avaliações de clientes insatisfeitos?
   -> problemas aparecem avaliações clientes insatisfeitos

Existem padrões de reclamação sobre a qualidade dos produtos?
   -> padrões reclamação sobre qualidade produtos

Qual foi o desempenho financeiro da Olist em 2018?
   -> desempenho financeiro olist 2018



In [3]:
def comparar(pergunta):
    print("=" * 110)
    print(pergunta)
    print("\n--- pergunta crua ---")
    imprimir_evidencias(recuperar(store, pergunta, processar=False))
    print("\n--- pergunta processada ---")
    imprimir_evidencias(recuperar(store, pergunta))

for p in perguntas:
    comparar(p)

Quais são os principais problemas relatados pelos clientes?

--- pergunta crua ---
 1. sim=0.866 | 1x | 1★×1 | As informações nos sites das lannister são irreais, dizem que os correios tiveram um imprevisto, mas em resposta quando questionados dizem que é transportadora. Devem nos passar informações corretas.
 2. sim=0.850 | 1x | 4★×1 | Somente um problema: todos os bonecos são musicais, no entanto, as músicas são em inglês.
 3. sim=0.846 | 1x | 3★×1 | Os produtos são bons
 4. sim=0.846 | 1x | 1★×1 | Quando os produtos eram vendidos e entregues pela própria Lojas lannister, chegavam sempre antes do prazo e sem problemas.
 5. sim=0.845 | 1x | 1★×1 | Os parceiros da stark não cumprem com seus prazos de entrega, e não prestam quaisquer assistência a seus clientes.
 6. sim=0.845 | 1x | 5★×1 | Os produtos vendidos através do stark são muito bons e sempre de qualidade.
 7. sim=0.845 | 1x | 5★×1 | Os produtos são ótimos!
 8. sim=0.844 | 1x | 4★×1 | Os fretes cobrados muitas vzs são maiores qu

a busca acerta bem o assunto, mas n distingue bem elogio de reclamação

uma opção seria testar a rescrita do processamento da pergunta com o a llm

In [3]:
from src.llm import criar_llm
from src.retrieval import recuperar_multiconsulta

llm = criar_llm()

def testar_multiconsulta(pergunta):
    print("=" * 110)
    print(pergunta)
    consultas, evidencias = recuperar_multiconsulta(store, pergunta, llm)
    print("\nBuscas geradas:")
    for i, c in enumerate(consultas, 1):
        print(f"  b{i}: {c}")
    print()
    imprimir_evidencias(evidencias, consultas)

for p in perguntas:
    testar_multiconsulta(p)

c:\dev\tech-challenge-fase4\venv\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.1) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
c:\dev\tech-challenge-fase4\venv\Lib\site-packages\langchain_google_genai\chat_models.py:49: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  from google.generativeai.caching import CachedContent  # type: ignore[import]


Quais são os principais problemas relatados pelos clientes?

Buscas geradas:
  b1: Entrega atrasou e não recebi no prazo
  b2: Produto veio quebrado e não funciona
  b3: Pedi reembolso e o suporte não responde
  b4: Qualidade péssima totalmente diferente da foto
  b5: Veio o modelo errado do produto

 1. sim=0.937 | 1x | 5★×1 | b1 | Inclusive recebi bem antes do prazo de entrega
 2. sim=0.953 | 1x | 1★×1 | b2 | O produto não funciona e veio errado.
 3. sim=0.915 | 1x | 1★×1 | b3 | Não recebi, pedi reembolso e o crédito não entrou na conta
 4. sim=0.922 | 1x | 1★×1 | b4 | Foto diferente do produto, qualidade ruim.
 5. sim=0.959 | 1x | 1★×1 | b5 | O produto enteque veio Errado
 6. sim=0.935 | 1x | 1★×1 | b1 | Já passou o prazo de entrega e ainda não recebi
 7. sim=0.952 | 1x | 1★×1 | b2 | O produto não veio funcionando
 8. sim=0.911 | 1x | 1★×1 | b3 | Pedi reembolso e sem resposta até momento
 9. sim=0.919 | 1x | 1★×1 | b4 | Produto totalmente diferente do anunciado ,péssima qualidade
10